In [11]:
import pandas as pd
import multiprocessing as mp
from ete3 import Tree
import pickle
from ete3 import NCBITaxa, Tree

In [12]:
import pickle
with open('outputs/connected_components.pickle', 'rb') as f:
    connected_components = pickle.load(f)
with open('outputs/connected_components_reps.pickle', 'rb') as f:
    representative = pickle.load(f)

In [13]:

def init_worker():
    global ncbi
    ncbi = NCBITaxa()
ncbi = NCBITaxa()


def get_taxid(species_name, prefer_rank="species"):
    try:
        name2taxids = ncbi.get_name_translator([species_name])
        if species_name not in name2taxids:
            if len(species_name.split(" "))>2:
                species_name=" ".join(species_name.split(" ")[0:2])
                get_ncbi_lineage_by_rank(" ".join(species_name.split(" ")[0:2]))
                try: 
                    return get_ncbi_lineage_by_rank(" ".join(species_name.split(" ")[0:2]))
                except:
                    print(species_name)
                    return 'None','None'
            raise ValueError(f"No NCBI taxid found for: {species_name!r}")
    
        candidate_taxids = name2taxids[species_name]
    
        if len(candidate_taxids) > 1:
            candidate_ranks = ncbi.get_rank(candidate_taxids)
            preferred = [
                taxid for taxid in candidate_taxids
                if candidate_ranks.get(taxid) == prefer_rank
            ]
            taxid = preferred[0] if preferred else candidate_taxids[0]
        else:
            taxid = candidate_taxids[0]
        
            return taxid
    except:
        print(species_name)

In [14]:

alldf=pd.read_csv("outputs/per_protein_diamond_data.tsv",sep="\t",index_col=0)
dipt_proteins=set(alldf[alldf.slineage.astype(str).str.contains('Diptera')]['accession'])
htgdf=alldf[alldf.HGT]
htgdf=htgdf.set_index('accession')
htgs=set(htgdf.index)

import pickle
with open('outputs/connected_components.pickle', 'rb') as f:
    connected_components = pickle.load(f)
with open('outputs/connected_components_reps.pickle', 'rb') as f:
    representative = pickle.load(f)

In [15]:
def get_sister_info(ranks):
    domain = kingdom = phylum = classt = order = None
    try:
        if 'Viruses' in str(ranks):
            domain='Viruses'
        else:
            domain=ranks['domain']
    except:
        superkingdom='None'
    try:
        kingdom=ranks['kingdom']
    except:
        kingdom='None'
    try:
        phylum=ranks['phylum']
    except:
        phylum='None'
    try:
        classt=ranks['class']
    except:
        classt='None'
    try:
        order=ranks['order']
    except:
        order='None'
    return [domain, kingdom, phylum, classt, order]


In [16]:
import os
tdo=[x for x in os.listdir("outputs/phylogenetic_dataset") if "rev_aa.contree" in os.listdir(f"outputs/phylogenetic_dataset/{x}")]  

In [29]:
def get_sister_names(i):
    try:
        t = Tree(f"outputs/phylogenetic_dataset/{i}/rev_aa_midpoint.treefile")
        
        with open(f"outputs/phylogenetic_dataset/{i}/phylo_dict.pkl", "rb") as f:
            lmap = pickle.load(f)
        
        
            
        for leaf in t:
            if leaf.name in lmap:
                l=lmap[leaf.name]
                leaf.add_features(sciarid= "Sciaridae" in str(l))
        
        mono=[]
        for node in t.get_monophyletic(values=[True], target_attr="sciarid"):
            mono.append(node)
        accepted_nodes=[]
        rejected_nodes=[]
        sister_names=set()
        for node in mono:
            accept = False
            sisters=[x for x in node.up if x.sciarid==False]
            for x in sisters:
                if lmap[x.name].split(";")[-1].strip()=='983967' or 'Cyberlindnera jadinii' in lmap[x.name].split(";")[-1].strip() or 'Metabacillus sp.' in lmap[x.name].split(";")[-1].strip():
                    print(x,lmap[x.name])
            sister_names=sister_names|set([lmap[x.name].split(";")[-1].strip() for x in sisters])
        return sister_names
    except:
        return set()

In [30]:
with mp.Pool(processes=19, initializer=init_worker) as pool:
    results = pool.map(get_sister_names, tdo)


--ODV84082.1 cellular organisms; Eukaryota; Opisthokonta; Fungi; Dikarya; Ascomycota; saccharomyceta; Saccharomycotina; Pichiomycetes; Pichiales; Pichiaceae; Ogataea; Ogataea/Candida clade; Ogataea arabinofermentans; 983967

--XP_020069085.1 cellular organisms; Eukaryota; Opisthokonta; Fungi; Dikarya; Ascomycota; saccharomyceta; Saccharomycotina; Saccharomycetes; Phaffomycetales; Phaffomycetaceae; Cyberlindnera; Cyberlindnera jadinii; Cyberlindnera jadinii NRRL Y-1542

--HZH60411.1 cellular organisms; Bacteria; Bacillati; Bacillota; Bacilli; Caryophanales; Bacillaceae; Metabacillus; unclassified Metabacillus; Metabacillus sp.

--HZH60411.1 cellular organisms; Bacteria; Bacillati; Bacillota; Bacilli; Caryophanales; Bacillaceae; Metabacillus; unclassified Metabacillus; Metabacillus sp.

--HZH60411.1 cellular organisms; Bacteria; Bacillati; Bacillota; Bacilli; Caryophanales; Bacillaceae; Metabacillus; unclassified Metabacillus; Metabacillus sp.

--WP_421650290.1 cellular organisms; Bacte

In [31]:
l=list(results)
all_names=list(set.union(*l))
with mp.Pool(processes=50, initializer=init_worker) as pool:
    tid_results = pool.map(get_taxid, all_names)


Cyberlindnera jadinii
983967
Metabacillus sp.


In [32]:
name_map={x:y for x,y in zip(all_names,tid_results)}

In [33]:
name_map['983967']=get_taxid('Ogataea arabinofermentans')
name_map['Metabacillus sp.']=get_taxid('Metabacillus')
name_map['Cyberlindnera jadinii']=get_taxid('Williopsis jadinii')

In [95]:
def get_sister_lcas(i):
    i=int(i)

    try:
        t = Tree(f"outputs/phylogenetic_dataset/{i}/rev_aa_midpoint.treefile")
        
        with open(f"outputs/phylogenetic_dataset/{i}/phylo_dict.pkl", "rb") as f:
            lmap = pickle.load(f)
        
        
            
        for leaf in t:
            if leaf.name in lmap:
                l=lmap[leaf.name]
                leaf.add_features(sciarid= "Sciaridae" in str(l))
        
        mono=[]
        for node in t.get_monophyletic(values=[True], target_attr="sciarid"):
        
            mono.append(node)
        accepted_nodes=[]
        rejected_nodes=[]
     
        n=1
        outputs=[]
        for node in mono:
            if len(set([x.name for x in node])&connected_components[i])>0:
              
                sisters=[x for x in node.up if x.sciarid==False]
                o=[f"{i}.{n}",str([x.name for x in node]),str(set([x.name for x in node])&set(connected_components[i]))]
                sister_tids=list(set([name_map[lmap[x.name].split(";")[-1].strip()] for x in sisters if lmap[x.name].split(";")[-1].strip() in name_map]))
                sister_tids=[x for x in sister_tids if x!=None]
                top=ncbi.get_topology(sister_tids)
                name=ncbi.get_taxid_translator([top.taxid])[top.taxid]
                o.extend([top.taxid, str(name)])
                l=ncbi.get_lineage(top.taxid)
                
                ranks={ncbi.get_rank([x])[x]:ncbi.get_taxid_translator([x])[x] for x in l}
                r=get_sister_info(ranks)
                o.extend(r)
                n+=1
                outputs.append(o)
        return  outputs
    except:
        print("fail",i)

In [96]:
with mp.Pool(processes=50, initializer=init_worker) as pool:
    tid_results = pool.map(get_sister_lcas, tdo)

fail 918


In [101]:
dfresult=pd.DataFrame()
for x in tid_results:

    if x !=None:
        for xi in x:
            dfresult.loc[float(xi[0]),['all_seqs','cluster_seq','taxid','name','domain','kingdom','phylum','class','order']]=xi[1:]

In [110]:
dfresult['tree_number']=[str(x).split(".")[0] for x in dfresult.index]
dfresult=dfresult.sort_index()

In [111]:
dfresult.to_csv("outputs/sister_automated_per_clade.tsv",sep='\t')

In [114]:
import ast
dfresult['cluster_seq']=[ast.literal_eval(x) for x in dfresult['cluster_seq']]

In [117]:
dfresult.explode('cluster_seq').to_csv('outputs/sister_automated_per_sequence.tsv',sep='\t')